# V26 — Solar Filament Real-Error Refinement + Counterfactual Selector

**Goal**: improve the **verified V25 public PQ = 0.39** without sacrificing the actual V25 winner. Full Kaggle notebook; official MAGFiLO train/test + **`SOLAR_V25_ALL_TRAINED_CHECKPOINTS.zip`** as inputs, GPU T4/T4×2, Internet to install missing packages. No external labeled solar datasets, no test labels.

**What actually changes:** load exact V20 YOLO11m detectors, V24 feature refiner, V25 curve proposal network, V25 69-feature rank ensemble and V25 real-mask refiner. Fine-tune a **NEW V26 refiner** on a larger, stratified set of real YOLO errors, including detector false positives; learn a **counterfactual model** that chooses between existing V25 and improved V26 masks based on expert-annotated calibration days. Evaluation is performed on separate dates. Outputs always include the untouched proven-configuration V25 baseline, new alternatives, score reports, and actual checkpoints.

**Caution:** The 104 detector-heldout images were repeatedly used in earlier V20–V25 iterations. A date-separated audit is useful but **not independent final generalization evidence**, and local pooled PQ is a proxy for competition PQ. Reaching 0.40 is an experiment, **not a promised score**. Full execution must occur on Kaggle GPU.

**Performance policy:** no training of old V20/V24/V25 models; V26 refiner can be reused from a prior attached V26 output. The test data are only passed to inference. Every trained checkpoint is written before test prediction and archived.

In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input');OUT=Path('/kaggle/working/solar_v26')
SEED=2026;GPU_ID=0;H=W=2048
RAW_CONF=.08;RAW_IOU=.72;MAX_DET=120
ALLOW_NON_V20_WEIGHT_HASH=False
FULL_VALIDATION_REQUIRED=True
CROP_SIDE=320
CURVE_WINDOW=768;CURVE_STRIDE=640;CURVE_SIDE=512;CURVE_MIN_AREA=170;CURVE_MAX_PROPOSALS=42
# New genuine-detector augmentation, not an architecture restart
V26_TRAIN_SCENES=260
V26_MAX_TRAIN_PAIRS=1700
V26_NEGATIVE_TARGET_FRACTION=.22
V26_EPOCHS=6
V26_BATCH=8
V26_STEPS_PER_EPOCH=185
V26_LEARNING_RATE=2.5e-5
V26_REQUIRE_OLD_WINNER=True
V25_EXPECTED_CSV_SHA='ac470fbfa0e2ca5f2d'   # V25 successful submission.csv, public 0.39
V25_EXPECTED_ROWS=1093
V25_EXPECTED_QUALITY_CUTOFF=.38
V26_SAVE_TOP_K=5
for directory in ('models','raw','refined','reports','submissions','curve_raw','real_pairs','action_cache','input_data'):
    (OUT/directory).mkdir(parents=True,exist_ok=True)
assert INPUT.is_dir(), 'Run this notebook on Kaggle; attach solar3k and SOLAR_V25_ALL_TRAINED_CHECKPOINTS.zip.'
print('V26 output:',OUT)

In [ ]:
import os,sys,time,json,hashlib,zipfile,shutil,gzip,random,subprocess,importlib.util
import gc as _gc_safe
from collections import defaultdict
import numpy as np,pandas as pd,cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from IPython.display import display

def require(module,package):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
require('pycocotools','pycocotools')
# Lock exact V22 version when available to maximize baseline reproduction.
try:
    from importlib.metadata import version as package_version
    installed_version=package_version('ultralytics')
except Exception:
    installed_version=None
if installed_version!='8.4.189':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.189'])
from pycocotools import mask as coco
from ultralytics import YOLO
import ultralytics,torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU T4/T4x2. No CPU or nano fallback.'
assert torch.cuda.device_count()>GPU_ID
print('Using',torch.cuda.get_device_name(GPU_ID),'ultralytics',ultralytics.__version__)
cv2.setNumThreads(0);random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
H=W=2048
EXT={'.jpeg','.jpg','.png','.tif','.tiff','.bmp'}
def files_in(d):
    p=Path(d)
    return sorted(q for q in p.iterdir() if q.is_file() and q.suffix.lower() in EXT) if p.is_dir() else []
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def save_json(p,obj):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_suffix(p.suffix+'.tmp');temp.write_text(json.dumps(obj,indent=2,default=str));temp.replace(p)
def enc(m):return coco.encode(np.asfortranarray(np.asarray(m,np.uint8)))
def txt(r):return r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else str(r['counts'])
def as_rle(x):return {'size':[H,W],'counts':x.encode('ascii') if isinstance(x,str) else x}
assert callable(_gc_safe.collect)
save_json(OUT/'reports'/'env.json',{'torch':torch.__version__,'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(GPU_ID),'no_prior_split_required':True,'prior_verified_public_PQ':0.39,'target_public_PQ':0.40})

## 1. Official data and protected acquisition-day partition

In [ ]:
ANAME='MAGFiLO_1.0_Annotations_kaggle2026_train.json'
found=sorted(INPUT.rglob(ANAME))
OFFICIAL=next((p.parent.parent for p in found if (p.parent.parent/'test'/'test_images').is_dir() and (p.parent.parent/'train'/'train_images').is_dir()),None)

def find_zip(name):
    matches=list(INPUT.rglob(name))
    return matches[0] if matches else None

def safe_images_unzip(archive,destination):
    destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            if not item.filename.lower().endswith(tuple(EXT)):continue
            target=destination/Path(item.filename).name
            if not target.exists():
                with z.open(item) as src,open(target,'wb') as dst:shutil.copyfileobj(src,dst,1<<20)
    return destination
if OFFICIAL:
    ANNOT=OFFICIAL/'train'/ANAME
    TRAIN={p.stem:p for p in files_in(OFFICIAL/'train'/'train_images')}
    TEST={p.stem:p for p in files_in(OFFICIAL/'test'/'test_images')}
    print('Using full official competition folder',OFFICIAL)
else:
    annot=found[0] if found else None
    ztest=find_zip('test.zip');ztrain=find_zip('sometrain.zip')
    assert annot is not None and ztest is not None, 'Attach official solar3k, or the annotation JSON and test.zip as Kaggle inputs.'
    ANNOT=annot
    TEST={p.stem:p for p in files_in(safe_images_unzip(ztest,OUT/'input_data'/'test_images'))}
    TRAIN={p.stem:p for p in files_in(safe_images_unzip(ztrain,OUT/'input_data'/'train_images'))} if ztrain else {}
    print('Using upload ZIP fallback. Validation may cover fewer than 104 physical images.')
assert len(TEST)==180,('Expected 180 test scenes; found',len(TEST))
assert len(set(TEST)&set(TRAIN))==0
DOC=json.loads(ANNOT.read_text())
BY_ID=defaultdict(list)
for ann in DOC['annotations']:BY_ID[str(ann['image_id'])].append(ann)
VERSIONS=defaultdict(list)
for item in DOC['images']:
    stem=Path(item['file_name']).stem
    if stem in TRAIN:VERSIONS[stem].append((str(item['id']),BY_ID.get(str(item['id']),[])))
assert set(TRAIN)==set(VERSIONS)
if TRAIN:
    days=defaultdict(list)
    for img in DOC['images']:
        stem=Path(img['file_name']).stem
        days[stem[:8]].append(stem)
    days={k:sorted(set(v)) for k,v in days.items()}
    shuffled=sorted(days)
    random.Random(SEED+2020).shuffle(shuffled)
    selected_days=[];n=0
    for d in shuffled:
        if n>=104:break
        selected_days.append(d);n+=len(days[d])
    VALIDATION=sorted(set(stem for d in selected_days for stem in days[d]) & set(TRAIN))
else:VALIDATION=[]
FULL_V20_VALIDATION=(len(VALIDATION)==104 and len(TRAIN)==707)
print('Images: train present',len(TRAIN),'test',len(TEST),'V20 tune available',len(VALIDATION),'full reproduction',FULL_V20_VALIDATION)
if VALIDATION:
    assert not ({s[:8] for s in VALIDATION} & {s[:8] for s in TRAIN if s not in VALIDATION})
save_json(OUT/'reports'/'dataset.json',{'annotation_sha256':sha(ANNOT),'train_available':len(TRAIN),
    'test_images':len(TEST),'validation_available':len(VALIDATION),'full_v20_reproduction':FULL_V20_VALIDATION,
    'validation_stems':VALIDATION,'test_ground_truth_used':False})

## 2. Recover original detector weights from your V25 trained-model ZIP

In [ ]:
EXPECT={
 'portable_global_m':('portable_global_m_best.pt','73fd3f3a4931'),
 'portable_native_m':('portable_native_m_best.pt','1a932d28e590'),
}
V20_DIR=Path('/kaggle/input/datasets/lighittha/solarouts/solar_v20_from_scratch/models')
ZIPS=sorted(set(p for p in INPUT.rglob('*.zip') if any(token in p.name.lower() for token in ('v20','v21','weights','models'))))
WEIGHTS={};HASHES={};SOURCES={}
(OUT/'models').mkdir(parents=True,exist_ok=True)
for model_name,(filename,prefix) in EXPECT.items():
    path=OUT/'models'/filename
    choices=[]
    direct=V20_DIR/filename
    if direct.is_file():choices.append(direct)
    choices+=[p for p in INPUT.rglob(filename) if p.is_file() and p!=direct]
    if path.is_file():choices.insert(0,path)
    if not path.is_file() or path.stat().st_size<20_000_000:
        restored=False
        for src in choices:
            if 30_000_000<src.stat().st_size<65_000_000:
                if src!=path:shutil.copy2(src,path)
                SOURCES[model_name]=str(src);restored=True;break
        if not restored:
            for archive in ZIPS:
                try:
                    with zipfile.ZipFile(archive) as z:
                        member='models/'+filename
                        if member not in z.namelist():continue
                        if not (30_000_000<z.getinfo(member).file_size<65_000_000):continue
                        temp=path.with_suffix('.part')
                        with z.open(member) as rf,open(temp,'wb') as wf:shutil.copyfileobj(rf,wf,1<<20)
                        temp.replace(path);SOURCES[model_name]=str(archive)+'::'+member
                        restored=True;break
                except (OSError,zipfile.BadZipFile) as e:print('Ignore corrupt ZIP:',archive,e)
        if not restored:raise FileNotFoundError('Missing trained V20 '+filename+f'. Expected at {V20_DIR/filename} or inside the V20 weights ZIP. This notebook never substitutes pretrained-only or nano models.')
    digest=sha(path)
    if not ALLOW_NON_V20_WEIGHT_HASH:
        assert digest.startswith(prefix),f'{filename}: wrong weights hash! Expected V20 prefix {prefix}, got {digest[:12]}'
    m=YOLO(str(path))
    n_params=sum(p.numel() for p in m.model.parameters())
    assert m.task=='segment' and 21_000_000<=n_params<=24_000_000,(model_name,m.task,n_params)
    del m;_gc_safe.collect()
    WEIGHTS[model_name]=path;HASHES[model_name]=digest
print('Recovered V20 model identity:',{k:v[:12] for k,v in HASHES.items()})
save_json(OUT/'reports'/'weights.json',{'sha256':HASHES,'source':SOURCES,'V20_checkpoint_hash_gate':not ALLOW_NON_V20_WEIGHT_HASH})

## 3. Locked V22 global/tiled source, exactly as used by V25

In [ ]:
# Locked V22 successful source definitions.
SRC_BASE_GLOBAL=('portable_global_m','g1536')
SRC_BASE_NATIVE=('portable_native_m','g1536')
SRC_HI=('portable_global_m','g1792')
SRC_TILES=('portable_native_m','t1280')
SRC_HIFLIP=('portable_global_m','g1792_hflip')
BASE_SOURCES=[SRC_HI,SRC_TILES]
EXTRA_SOURCES=[]
ACTIVE=BASE_SOURCES+EXTRA_SOURCES
ENGINES={}
def get_engine(name):
    if name not in ENGINES:
        ENGINES.clear();_gc_safe.collect();torch.cuda.empty_cache()
        ENGINES[name]=YOLO(str(WEIGHTS[name]))
    return ENGINES[name]

def view_windows(view):
    if view=='g1536':return [(0,0,2048,1536,False)]
    if view=='g1792':return [(0,0,2048,1792,False)]
    if view=='g1792_hflip':return [(0,0,2048,1792,True)]
    if view=='t1280':return [(x,y,1280,1024,False) for y in (0,768) for x in (0,768)]
    raise ValueError(view)
def sig(source):
    name,view=source
    return hashlib.sha256(json.dumps({'model_sha256':HASHES[name],'view':view,
      'conf':RAW_CONF,'iou':RAW_IOU,'max_det':MAX_DET,'retina_masks':True,
      'agnostic_nms':True,'ultralytics':ultralytics.__version__,'version':'V22_1' if view in ('g1792','t1280','g1536') else 'V24_1'},sort_keys=True).encode()).hexdigest()
def cache_file(source,stem):
    name,view=source
    return OUT/'raw'/name/view/(stem+'.json.gz')
@torch.inference_mode()
def raw_predictions(source,stem):
    path=cache_file(source,stem);identifier=sig(source)
    if path.exists():
        try:
            with gzip.open(path,'rt') as f:v=json.load(f)
            if v.get('signature')==identifier and v.get('stem')==stem:return v['predictions']
        except (OSError,ValueError,KeyError):pass
    image_path=TRAIN.get(stem) or TEST.get(stem)
    if image_path is None:raise FileNotFoundError('Missing scene: '+stem)
    gray=cv2.imread(str(image_path),cv2.IMREAD_GRAYSCALE)
    assert gray is not None and gray.shape==(H,W),(stem, None if gray is None else gray.shape)
    predictions=[]
    name,view=source
    model=get_engine(name)
    for x,y,side,net,hflip in view_windows(view):
        roi=np.ascontiguousarray(gray[y:y+side,x:x+side])
        if hflip:roi=cv2.flip(roi,1)
        image=cv2.cvtColor(roi,cv2.COLOR_GRAY2BGR)
        result=model.predict(image,imgsz=net,conf=RAW_CONF,iou=RAW_IOU,max_det=MAX_DET,
                             agnostic_nms=True,retina_masks=True,device=GPU_ID,verbose=False)[0]
        if result.masks is None or result.boxes is None:continue
        mask_data=result.masks.data.detach().cpu().numpy()
        confidences=result.boxes.conf.detach().cpu().numpy()
        for mask_arr,conf in zip(mask_data,confidences):
            mask=(mask_arr>0.5).astype(np.uint8)
            if mask.shape!=(side,side):mask=cv2.resize(mask,(side,side),interpolation=cv2.INTER_NEAREST)
            if hflip:mask=np.ascontiguousarray(cv2.flip(mask,1))
            if int(mask.sum())<100:continue
            if side!=2048:
                ys,xs=np.where(mask)
                truncated=bool((x>0 and xs.min()<10) or (x+side<W and xs.max()>=side-10) or
                               (y>0 and ys.min()<10) or (y+side<H and ys.max()>=side-10))
                full=np.zeros((H,W),np.uint8)
                full[y:y+side,x:x+side]=mask
            else:truncated=False;full=mask
            rr=enc(full)
            predictions.append({'counts':txt(rr),'score':float(conf),'area':int(coco.area(rr)),
                                'truncated':truncated,'source_view':view})
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with gzip.open(tmp,'wt',compresslevel=3) as f:
        json.dump({'signature':identifier,'stem':stem,'view':view,'predictions':predictions},f,separators=(',',':'))
    tmp.replace(path)
    return predictions
print('V23 views including immutable winner:',ACTIVE)

In [ ]:
def recipe(name,sources,gconf=.24,tconf=.32,area=200,seam=False,refine='none',minfree=.45):
    return {'name':name,'sources':[list(s) for s in sources],'global_conf':float(gconf),
            'tile_conf':float(tconf),'min_area':int(area),'seam_guard':bool(seam),
            'nms_iou':.50,'tile_bonus':.08,'min_free':float(minfree),'refine':refine}

# This exact configuration produced V22_ALT_2.csv (public PQ 0.37).
WINNER=recipe('V22_ALT2_WINNER_037',[SRC_HI,SRC_TILES],
              gconf=.24,tconf=.42,area=200,seam=True,refine='none',minfree=.45)
WINNER['nms_iou']=.50
WINNER['tile_bonus']=.08
WINNER['engine']='v22'
WINNER['fusion']='none'
WINNER['tile_policy']='baseline'

def experiment(name,sources=None,gconf=.24,tconf=.42,area=200,seam=True,
               bonus=.08,nms=.50,minfree=.45,tile_policy='baseline',fusion='none'):
    r=recipe(name,sources if sources is not None else [SRC_HI,SRC_TILES],
             gconf=gconf,tconf=tconf,area=area,seam=seam,minfree=minfree)
    r.update(engine='v23',tile_bonus=bonus,nms_iou=nms,tile_policy=tile_policy,fusion=fusion)
    return r


def modify_rle(rr,operation):
    if operation=='none':return rr
    box=coco.toBbox(rr);x,y,w,h=[int(q) for q in box]
    if w<1 or h<1:return rr
    pad=3;x0=max(0,x-pad);y0=max(0,y-pad);x1=min(W,x+w+pad);y1=min(H,y+h+pad)
    full=coco.decode(rr).astype(np.uint8)
    roi=full[y0:y1,x0:x1]
    if operation=='close3':
        result=cv2.morphologyEx(roi,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
    elif operation=='dilate1':
        result=cv2.dilate(roi,np.ones((3,3),np.uint8),iterations=1)
    elif operation=='erode1':
        result=cv2.erode(roi,np.ones((3,3),np.uint8),iterations=1)
    else:raise ValueError(operation)
    if int(result.sum())<50:return rr
    full[y0:y1,x0:x1]=result
    return enc(full)

def select_masks(raw,cfg):
    sources={tuple(pair) for pair in cfg['sources']}
    candidates=[]
    for source,items in raw.items():
        if source not in sources:continue
        threshold=cfg['tile_conf'] if source[1].startswith('t') else cfg['global_conf']
        for p in items:
            score=float(p['score']);area=int(p['area'])
            if score<threshold or area<cfg['min_area']:continue
            if cfg['seam_guard'] and source[1].startswith('t') and p.get('truncated',False):continue
            rank=score+(cfg['tile_bonus'] if source[1].startswith('t') else 0.)
            candidates.append((rank,area,p))
    candidates.sort(key=lambda q:(-q[0],-q[1]))
    masks=[];originals=[];occupied=np.zeros((H,W),np.uint8)
    for rank,area,p in candidates:
        rr=as_rle(p['counts'])
        if originals:
            ov=np.asarray(coco.iou([rr],originals,[0]*len(originals)))[0]
            if np.any(ov>=cfg['nms_iou']):continue
        mask=coco.decode(rr).astype(np.uint8)
        free=mask&(occupied==0)
        free_area=int(free.sum())
        if free_area<cfg['min_area'] or free_area/max(1,area)<cfg['min_free']:continue
        cleaned=enc(free)
        if cfg.get('refine','none')!='none':cleaned=modify_rle(cleaned,cfg['refine'])
        # Retain exact V20 occupied behavior for baseline; for refined recipes use actual refined footprint.
        if cfg.get('refine','none')!='none':
            shaped=coco.decode(cleaned).astype(np.uint8)&(occupied==0)
            if int(shaped.sum())<cfg['min_area']:continue
            cleaned=enc(shaped)
            occupied|=shaped
        else:occupied|=free
        masks.append(cleaned);originals.append(rr)
        if len(masks)>=120:break
    return masks

# Do not change the winner branch; these tests protect the baseline.
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['global_conf']==.24 and WINNER['tile_conf']==.42
assert WINNER['seam_guard'] and WINNER['min_area']==200
assert WINNER['nms_iou']==.5 and WINNER['tile_bonus']==.08
assert callable(_gc_safe.collect)

## 4. Historical pooled-PQ proxy, expert by expert

In [ ]:
def annotation_rle(a):
    seg=a.get('segmentation')
    if seg is None or seg==[]:return None
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):
            obj=coco.frPyObjects(seg,H,W)
            return coco.merge(obj) if isinstance(obj,list) else obj
        rr=dict(seg)
        if isinstance(rr.get('counts'),str):rr['counts']=rr['counts'].encode('ascii')
        return rr
    if len(seg)>0 and isinstance(seg[0],(int,float)):seg=[seg]
    polygons=[p for p in seg if len(p)>=6 and len(p)%2==0]
    if not polygons:return None
    rr=coco.frPyObjects(polygons,H,W)
    return coco.merge(rr) if isinstance(rr,list) else rr
TRUTH={}
def gt(stem):
    if stem not in TRUTH:
        TRUTH[stem]=[[rr for a in arr if (rr:=annotation_rle(a)) is not None]
                     for _,arr in VERSIONS[stem]]
    return TRUTH[stem]
def match_counts(truth,preds):
    if not truth:return np.array([0.,0.,float(len(preds)),0.])
    if not preds:return np.array([0.,0.,0.,float(len(truth))])
    ious=np.asarray(coco.iou(truth,preds,[0]*len(preds)),float)
    # Match the earlier V20 PQ proxy EXACTLY for control-score parity.
    ii,jj=linear_sum_assignment(-ious)
    ok=ious[ii,jj]>.5
    tp=int(ok.sum())
    return np.array([float(ious[ii[ok],jj[ok]].sum()),float(tp),
                     float(len(preds)-tp),float(len(truth)-tp)])
def physical_counts(stem,preds):
    result=np.zeros(4,float)
    for version in gt(stem):result+=match_counts(version,preds)
    return result
def pq(count):
    den=count[1]+.5*(count[2]+count[3])
    return float(count[0]/den) if den>0 else 0.
def metrics(count):
    den=count[1]+.5*(count[2]+count[3])
    return {'PQ':pq(count),'SQ':float(count[0]/count[1]) if count[1] else 0.,
       'RQ':float(count[1]/den) if den else 0.,'TP':int(count[1]),
       'FP':int(count[2]),'FN':int(count[3])}
example=np.zeros((H,W),np.uint8);example[40:61,50:92]=1
assert coco.area(as_rle(txt(enc(example))))==example.sum()
assert abs(pq(match_counts([enc(example)],[enc(example)]))-1)<1e-9
print('PQ scorer and full-size RLE roundtrip: PASS')

## 5. Restore frozen V24 and V25 networks and model assets

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import joblib, math, time, io
from torchvision.models import resnet18

class DoubleConv(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.block=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
            nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x):return self.block(x)

class PromptUNet(nn.Module):
    def __init__(self):
        super().__init__();e=resnet18(weights=None);old=e.conv1
        e.conv1=nn.Conv2d(4,64,7,2,3,bias=False)
        self.stem=nn.Sequential(e.conv1,e.bn1,e.relu);self.pool=e.maxpool
        self.e1=e.layer1;self.e2=e.layer2;self.e3=e.layer3;self.e4=e.layer4
        self.d4=DoubleConv(768,192);self.d3=DoubleConv(320,128)
        self.d2=DoubleConv(192,96);self.d1=DoubleConv(160,64)
        self.out=nn.Sequential(nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                               nn.Conv2d(64,32,3,padding=1),nn.ReLU(inplace=True),nn.Conv2d(32,1,1))
    def forward(self,x):
        e0=self.stem(x);e1=self.e1(self.pool(e0));e2=self.e2(e1);e3=self.e3(e2);e4=self.e4(e3)
        def up(a,b):return torch.cat([F.interpolate(a,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1)
        a=self.d4(up(e4,e3));a=self.d3(up(a,e2));a=self.d2(up(a,e1));a=self.d1(up(a,e0))
        return self.out(a)

def make_channels(gray_uint8,mask_uint8):
    gray=np.asarray(gray_uint8,dtype=np.uint8)
    equal=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(gray)
    blur=cv2.GaussianBlur(gray,(3,3),0)
    dx=cv2.Sobel(blur,cv2.CV_32F,1,0,ksize=3);dy=cv2.Sobel(blur,cv2.CV_32F,0,1,ksize=3)
    edge=np.clip(cv2.magnitude(dx,dy)/240.,0,1)
    return np.stack([gray/255.,equal/255.,mask_uint8.astype(np.float32),edge],axis=0).astype(np.float32)

def normalized_bbox_roi(box,scale=1.48,min_extent=120):
    x,y,w,h=map(float,box)
    if w<=0 or h<=0:return None
    dx=min(W,max(float(min_extent),w*scale));dy=min(H,max(float(min_extent),h*scale))
    cx=x+w/2;cy=y+h/2
    x0=int(np.clip(round(cx-dx/2),0,W-1));y0=int(np.clip(round(cy-dy/2),0,H-1))
    x1=int(np.clip(round(cx+dx/2),x0+1,W));y1=int(np.clip(round(cy+dy/2),y0+1,H))
    return x0,y0,x1,y1

def find_original_asset(filename):
    direct=sorted(INPUT.rglob(filename))
    for p in direct:
        if p.is_file() and p.stat().st_size>100:return p.read_bytes(),str(p)
    for z in sorted(INPUT.rglob('*.zip')):
        if not any(token in z.name.lower() for token in ('v24','model','trained')):continue
        try:
            with zipfile.ZipFile(z) as arc:
                matches=[m for m in arc.namelist() if Path(m).name==filename]
                if matches:return arc.read(matches[0]),str(z)+'::'+matches[0]
                # V24 full output bundles a second ZIP containing its scored CSVs.
                for nested in [m for m in arc.namelist() if m.lower().endswith('.zip')]:
                    try:
                        with zipfile.ZipFile(io.BytesIO(arc.read(nested))) as inner:
                            hits=[x for x in inner.namelist() if Path(x).name==filename]
                            if hits:return inner.read(hits[0]),str(z)+'::'+nested+'::'+hits[0]
                    except (OSError,zipfile.BadZipFile):pass
        except (OSError,zipfile.BadZipFile):continue
    return None,None

for filename in ('SOLAR_V24_REFINER_BEST.pt','SOLAR_V24_INSTANCE_QUALITY.joblib'):
    target=OUT/'models'/filename
    data,source=find_original_asset(filename)
    if data is None:
        if REQUIRE_V24_MODELS:raise FileNotFoundError('Attach V24 model archive; missing '+filename)
    else:
        if not target.exists() or sha(target)!=hashlib.sha256(data).hexdigest():target.write_bytes(data)
        print('Recovered V24 asset:',filename, 'from',source,'SHA:',sha(target)[:16])

V24_REFINER_PATH=OUT/'models'/'SOLAR_V24_REFINER_BEST.pt'
V24_QUALITY_PATH=OUT/'models'/'SOLAR_V24_INSTANCE_QUALITY.joblib'
refiner_v24=PromptUNet().to('cuda:'+str(GPU_ID))
refiner_v24.load_state_dict(torch.load(V24_REFINER_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_v24.eval()
meta_v24=joblib.load(V24_QUALITY_PATH)
quality_v24=meta_v24['model'];V24_FEATURES=meta_v24['features']
assert len(V24_FEATURES)==15 and callable(quality_v24.predict_proba)
print('Recovered genuine V24 quality model, OOF AUROC:',meta_v24.get('oof_auc'))

In [ ]:
def matching_scores(rr,records):
    if not records:return 0.,0.,0.
    rles=[as_rle(q['counts']) for q in records]
    ious=np.asarray(coco.iou([rr],rles,[0]*len(rles)),float).reshape(-1)
    i=int(ious.argmax());return float(ious[i]),float(records[i]['score']),float(max(q['score'] for q in records))

@torch.inference_mode()
def run_refiner(model,gray,rr,threshold=.50,min_prompt_iou=.54):
    crop=normalized_bbox_roi([int(v) for v in coco.toBbox(rr)],scale=1.48)
    if crop is None:return rr
    x0,y0,x1,y1=crop;original=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    if not original.any():return rr
    image=cv2.resize(gray[y0:y1,x0:x1],(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_AREA)
    prompt=cv2.resize(original,(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_NEAREST)
    inp=torch.from_numpy(make_channels(image,prompt))[None].cuda(GPU_ID)
    with torch.autocast('cuda',dtype=torch.float16):
        probabilities=model(inp).sigmoid()[0,0].float().cpu().numpy()
    refined=(cv2.resize(probabilities,(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)>=threshold).astype(np.uint8)
    oa=int(original.sum());na=int(refined.sum())
    if na<80 or not (.52<=na/max(oa,1)<=1.65):return rr
    overlap=int((refined&original).sum())/max(1,oa+na-int((refined&original).sum()))
    if overlap<min_prompt_iou:return rr
    full=np.zeros((H,W),np.uint8);full[y0:y1,x0:x1]=refined
    return enc(full)

def load_base(stem):
    raw={SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    return raw,select_masks(raw,WINNER)

def v24_refined_masks(stem,baseline,gray=None):
    key=sha(V24_REFINER_PATH)[:18];p=OUT/'refined'/('v24_'+key+'_'+stem+'.json.gz')
    if p.is_file():
        with gzip.open(p,'rt') as f:v=json.load(f)
        if len(v)==len(baseline):return [as_rle(x) for x in v]
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    proposed=[run_refiner(refiner_v24,gray,rr) for rr in baseline]
    occupied=np.zeros((H,W),np.uint8);final=[]
    for a,b in zip(baseline,proposed):
        f=coco.decode(b).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:f=coco.decode(a).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:continue
        final.append(enc(f));occupied|=f
    if len(final)!=len(baseline):final=baseline
    p.parent.mkdir(parents=True,exist_ok=True)
    with gzip.open(p,'wt',compresslevel=3) as f:json.dump([txt(v) for v in final],f)
    return final

def features_v24(stem,base,refined,gray=None,raw=None):
    raw=raw if raw is not None else {SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    rows=[]
    for rr,new in zip(base,refined):
        x,y,w,h=map(float,coco.toBbox(rr));mask=coco.decode(rr).astype(np.uint8);area=int(mask.sum())
        dilation=cv2.dilate(mask,np.ones((11,11),np.uint8))
        ring=(dilation>0)&(mask==0)
        inside=float(gray[mask>0].mean()) if area else 0.;outside=float(gray[ring].mean()) if ring.any() else inside
        ga,gscore,gmax=matching_scores(rr,raw[SRC_HI]);ta,tscore,tmax=matching_scores(rr,raw[SRC_TILES])
        na=float(coco.area(new));oa=float(coco.area(rr))
        riou=float(coco.iou([rr],[new],[0])[0,0])
        rows.append({'base_score':max(gscore,tscore),'base_area_log':float(np.log1p(area)),
                     'base_box_ar':float(np.log((w+1)/(h+1))),'base_fill':float(area/max(1,w*h)),
                     'base_center_x':float((x+w/2)/W),'base_center_y':float((y+h/2)/H),
                     'global_agreement':ga,'tile_agreement':ta,'global_maxconf':gmax,'tile_maxconf':tmax,
                     'pixel_avg_inside':inside/255.,'pixel_avg_ring':outside/255.,
                     'contrast':(outside-inside)/255.,'refined_iou':riou,'refined_area_ratio':na/max(oa,1.)})
    return pd.DataFrame(rows,columns=V24_FEATURES)

def v24_control(stem):
    raw,base=load_base(stem)
    if not base:return []
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    refined=v24_refined_masks(stem,base,gray)
    df=features_v24(stem,base,refined,gray,raw)
    qs=quality_v24.predict_proba(df[V24_FEATURES].fillna(0.))[:,1]
    return [rr for rr,q in zip(base,qs) if q>=.25]  # Exact V24 QUALITY_025: no extra clipping

s=np.zeros((H,W),np.uint8);s[50:90,70:130]=1
assert int(coco.area(enc(s)))==2400
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['tile_conf']==.42 and WINNER['global_conf']==.24 and WINNER['seam_guard']
print('V24 control frozen. New V25 scoring cannot alter its thresholds or masks.')

In [ ]:
# Restore all genuinely trained V25 checkpoint bytes; no substitute/pretrained-only networks.
from skimage.morphology import skeletonize
from skimage.segmentation import watershed

V25_REAL_PATH=OUT/'models'/'SOLAR_V25_REAL_REFINER_BEST.pt'
V25_CURVE_PATH=OUT/'models'/'SOLAR_V25_CURVE_BEST.pt'
V25_QUALITY_PATH=OUT/'models'/'SOLAR_V25_EXTENDED_QUALITY.joblib'
for p in (V25_REAL_PATH,V25_CURVE_PATH,V25_QUALITY_PATH):
    data,source=find_original_asset(p.name)
    if data is None:raise FileNotFoundError('Missing '+p.name+'. Attach the complete V25 trained checkpoint ZIP.')
    if not p.exists() or sha(p)!=hashlib.sha256(data).hexdigest():p.write_bytes(data)
    print('Recovered frozen V25',p.name,sha(p)[:18], 'from',source)
refiner_real=PromptUNet().cuda(GPU_ID)
refiner_real.load_state_dict(torch.load(V25_REAL_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_real.eval();REAL_REFINER_SHA=sha(V25_REAL_PATH)

def real_refine(stem,rr,gray=None):
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    return run_refiner(refiner_real,gray,rr,threshold=.5,min_prompt_iou=.50)

v25_ranker_package=joblib.load(V25_QUALITY_PATH)
new_rankers=v25_ranker_package['models']
V25_QUALITY_CUTOFF=float(v25_ranker_package['cutoff'])
print('Frozen V25 ranker cutoff:',V25_QUALITY_CUTOFF,'feature count:',len(v25_ranker_package['columns']))
assert abs(V25_QUALITY_CUTOFF-V25_EXPECTED_QUALITY_CUTOFF)<1e-6, 'Wrong V25 ranking checkpoint/cutoff'

def rank_probs(models,df):
    if df.empty:return np.asarray([],float)
    a=df[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)
    return np.mean([m.predict_proba(a)[:,1] for m in models],axis=0)


In [ ]:
class MiniUNet(nn.Module):
    def __init__(self,channels=2,outputs=3):
        super().__init__()
        self.a=DoubleConv(channels,24);self.b=DoubleConv(24,48);self.c=DoubleConv(48,96)
        self.d=DoubleConv(96,160)
        self.up3=DoubleConv(160+96,96);self.up2=DoubleConv(96+48,64)
        self.up1=DoubleConv(64+24,32);self.final=nn.Conv2d(32,outputs,1)
    def forward(self,x):
        a=self.a(x);b=self.b(F.avg_pool2d(a,2));c=self.c(F.avg_pool2d(b,2));d=self.d(F.avg_pool2d(c,2))
        u=self.up3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode='bilinear',align_corners=False),c],1))
        u=self.up2(torch.cat([F.interpolate(u,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        u=self.up1(torch.cat([F.interpolate(u,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return self.final(u)


curve_path=V25_CURVE_PATH
curve_net=MiniUNet().cuda(GPU_ID)
curve_net.load_state_dict(torch.load(curve_path,map_location='cpu',weights_only=False)['model'],strict=True)
curve_net.eval();CURVE_SHA=sha(curve_path)
print('Loaded frozen V25 curve network',CURVE_SHA[:18])

## 6. V25 curve proposals are a frozen *feature context*, not an untested default addition

In [ ]:
def curve_locations():
    starts=list(range(0,H-CURVE_WINDOW+1,CURVE_STRIDE))
    if starts[-1]!=H-CURVE_WINDOW:starts.append(H-CURVE_WINDOW)
    return [(x,y) for y in starts for x in starts]
CURVE_LOCATIONS=curve_locations()
assert len(CURVE_LOCATIONS)==9, CURVE_LOCATIONS

def curve_input(gray):
    equ=cv2.createCLAHE(clipLimit=2.,tileGridSize=(8,8)).apply(gray)
    return torch.from_numpy(np.stack([gray/255.,equ/255.],0).astype(np.float32))[None].cuda(GPU_ID)

@torch.inference_mode()
def curve_proposals(stem):
    cache=OUT/'curve_raw'/(stem+'.json.gz')
    signature=hashlib.sha256((CURVE_SHA+stem+str(CURVE_WINDOW)+str(CURVE_SIDE)).encode()).hexdigest()
    if cache.exists():
        try:
            with gzip.open(cache,'rt') as f:v=json.load(f)
            if v.get('identity')==signature:return v['proposals']
        except (ValueError,OSError,KeyError):pass
    image=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    if image is None:raise FileNotFoundError(stem)
    props=[]
    for x0,y0 in CURVE_LOCATIONS:
        raw=image[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]
        inp=cv2.resize(raw,(CURVE_SIDE,CURVE_SIDE),interpolation=cv2.INTER_AREA)
        with torch.autocast('cuda',dtype=torch.float16):
            probs=curve_net(curve_input(inp)).sigmoid()[0].float().cpu().numpy()
        fg=cv2.resize(probs[0],(CURVE_WINDOW,CURVE_WINDOW))
        center=cv2.resize(probs[1],(CURVE_WINDOW,CURVE_WINDOW))
        seg=(fg>.53).astype(np.uint8)
        n,components,stats,_=cv2.connectedComponentsWithStats(seg,connectivity=8)
        for k in range(1,n):
            component_area=int(stats[k,cv2.CC_STAT_AREA])
            if component_area<CURVE_MIN_AREA:continue
            local=(components==k).astype(np.uint8)
            # Conservatively split only when two *distinct* centerline islands exist.
            line=(center>.56).astype(np.uint8)&local
            m,markers,mark_stats,_=cv2.connectedComponentsWithStats(line,connectivity=8)
            good=[q for q in range(1,m) if mark_stats[q,cv2.CC_STAT_AREA]>=8]
            pieces=[local]
            if 2<=len(good)<=5 and component_area>700:
                seeds=np.zeros_like(local,np.int32)
                for label_idx,mark_idx in enumerate(good,1):seeds[markers==mark_idx]=label_idx
                regions=watershed(-cv2.distanceTransform(local,cv2.DIST_L2,3),seeds,mask=(local>0))
                candidates=[(regions==z).astype(np.uint8) for z in range(1,len(good)+1)]
                if all(int(c.sum())>=CURVE_MIN_AREA for c in candidates):pieces=candidates
            for part in pieces:
                area=int(part.sum())
                if area<CURVE_MIN_AREA:continue
                sy,sx=np.where(part)
                if min(sx)<8 and x0>0 or max(sx)>CURVE_WINDOW-9 and x0+CURVE_WINDOW<W:continue
                if min(sy)<8 and y0>0 or max(sy)>CURVE_WINDOW-9 and y0+CURVE_WINDOW<H:continue
                local_gray=raw[part>0];mask_score=float(np.mean(fg[part>0]))
                line_score=float(np.mean(center[part>0]))
                if mask_score<.59 or line_score<.18:continue
                # Relatively dark features, not arbitrary bright solar active-region structures.
                patch=raw[max(0,sy.min()-16):min(CURVE_WINDOW,sy.max()+17),
                          max(0,sx.min()-16):min(CURVE_WINDOW,sx.max()+17)]
                if len(local_gray) and float(np.median(local_gray))>float(np.percentile(patch,75))+8:continue
                full=np.zeros((H,W),np.uint8)
                full[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]=part
                rr=enc(full)
                props.append({'counts':txt(rr),'score':float(np.clip(.65*mask_score+.35*line_score,0,1)),
                              'area':int(coco.area(rr)),'truncated':False,'source_view':'curve_v25',
                              'center_support':line_score})
    props.sort(key=lambda p:p['score'],reverse=True)
    distinct=[];rles=[]
    for p in props:
        rr=as_rle(p['counts'])
        if rles and float(np.max(coco.iou([rr],rles,[0]*len(rles))))>.33:continue
        distinct.append(p);rles.append(rr)
        if len(distinct)>=CURVE_MAX_PROPOSALS:break
    with gzip.open(cache,'wt',compresslevel=3) as f:
        json.dump({'identity':signature,'stem':stem,'proposals':distinct},f,separators=(',',':'))
    return distinct

def extend_with_curve(stem,baseline,raw=None):
    # Existing instance identities are never silently split or replaced here.
    raw=raw if raw is not None else load_base(stem)[0]
    base_rles=list(baseline)
    base_all=base_rles.copy()
    occupied=np.zeros((H,W),np.uint8)
    for rr in base_rles:occupied|=coco.decode(rr).astype(np.uint8)
    additions=[]
    for p in curve_proposals(stem):
        rr=as_rle(p['counts'])
        if base_all and float(np.max(coco.iou([rr],base_all,[0]*len(base_all))))>=.25:continue
        part=coco.decode(rr).astype(np.uint8)
        free=part&(occupied==0)
        if int(free.sum())<CURVE_MIN_AREA or free.sum()/max(1,part.sum())<.70:continue
        accepted=enc(free)
        additions.append(accepted);occupied|=free;base_all.append(accepted)
    return base_rles+additions,['detector']*len(base_rles)+['curve']*len(additions)

print('Curve candidate generation initialized:',len(CURVE_LOCATIONS),'windows per 2048 scene')

## 7. The exact V25 69-feature representation and its frozen ensemble

In [ ]:
from scipy.spatial.distance import cdist

# These 40+ new features are computed from *predicted masks/images only*.
EXTRA_FIELDS=['log_perimeter','log_convex_area','solidity','extent','elongation','eccentricity',
 'aspect_absolute','area_frac_box','edge_density','endpoint_count','branchpoint_count','skeleton_length',
 'skeleton_area_ratio','mean_thickness','max_thickness','width_variation','local_component_count',
 'intensity_inside_std','intensity_inside_p10','intensity_inside_p50','intensity_inside_p90',
 'intensity_ring_p50','intensity_diff_p10','intensity_diff_mean','inside_entropy','sobel_inside_mean',
 'sobel_ring_mean','contrast_normalized','sun_radial','rim_distance','mask_near_solar_limb',
 'hull_vs_mask','compactness','hu_1','hu_2','pca_angle_sin','pca_angle_cos',
 'neighbor_center_dist','neighbor_bbox_iou','neighbor_area_ratio','n_neighbors_100',
 'curve_origin','curve_conf','curve_center_support','curve_vs_global_iou','curve_vs_tile_iou',
 'raw_global_count','raw_tile_count','raw_high_conf_global_count','raw_high_conf_tile_count',
 'mask_global_rank','mask_tile_rank','mask_area_over_disk','intensity_outside_std']
NEW_FEATURES=list(V24_FEATURES)+EXTRA_FIELDS
assert len(NEW_FEATURES)>60

def image_entropy(pixels):
    if len(pixels)==0:return 0.
    hist=np.bincount((pixels//16).astype(np.uint8),minlength=16).astype(float)
    hist=hist/max(hist.sum(),1);h=hist[hist>0]
    return float(-(h*np.log(h)).sum())

def extract_extra(gray,rr,all_boxes,source_kind,curve_meta,raw):
    x,y,w,h=map(float,coco.toBbox(rr));x0=max(0,int(x)-12);y0=max(0,int(y)-12)
    x1=min(W,int(np.ceil(x+w))+12);y1=min(H,int(np.ceil(y+h))+12)
    # COCO decode is full-frame but geometric operations are limited to a small crop.
    mask=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    g=gray[y0:y1,x0:x1];area=int(mask.sum());yy,xx=np.where(mask>0)
    ring=cv2.dilate(mask,np.ones((9,9),np.uint8))&(1-mask)
    inside=g[mask>0];outside=g[ring>0]
    if not len(inside):inside=np.asarray([0],np.uint8)
    if not len(outside):outside=inside
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    perimeter=float(sum(cv2.arcLength(c,True) for c in contours))
    hull_area=float(sum(cv2.contourArea(cv2.convexHull(c)) for c in contours if len(c)>=3))
    # Downsample skeleton for speed; normalize pixel-to-image ratio.
    ratio=min(1.,150/max(mask.shape));sw=max(8,int(mask.shape[1]*ratio));sh=max(8,int(mask.shape[0]*ratio))
    reduced=(cv2.resize(mask,(sw,sh),interpolation=cv2.INTER_NEAREST)>0).astype(np.uint8)
    sk=skeletonize(reduced>0).astype(np.uint8)
    neigh=cv2.filter2D(sk,cv2.CV_16S,np.ones((3,3),np.uint8),borderType=cv2.BORDER_CONSTANT)-sk.astype(np.int16)
    endpoints=int(((sk>0)&(neigh==1)).sum());branches=int(((sk>0)&(neigh>=3)).sum())
    dist=cv2.distanceTransform(reduced,cv2.DIST_L2,3)[reduced>0]
    components=cv2.connectedComponents(reduced,connectivity=8)[0]-1
    if len(xx)>1:
        cov=np.cov(np.stack([xx,yy]).astype(float));ev,vec=np.linalg.eigh(cov)
        major=vec[:,np.argmax(ev)];elong=float(np.sqrt((max(ev)+1)/(min(ev)+1)))
        ecc=float(np.sqrt(max(0.,1-min(ev)/(max(ev)+1e-6))))
    else:major=np.array([1.,0.]);elong=1.;ecc=0.
    sob=cv2.magnitude(cv2.Sobel(g,cv2.CV_32F,1,0),cv2.Sobel(g,cv2.CV_32F,0,1))
    hu=cv2.HuMoments(cv2.moments(mask)).flatten()
    cx=x+w/2;cy=y+h/2;rad=np.hypot(cx-W/2,cy-H/2)/(W/2)
    other_boxes=np.asarray(all_boxes,float).reshape(-1,4)
    current_centroid=np.array([cx,cy]);centroids=other_boxes[:,:2]+other_boxes[:,2:]/2 if len(other_boxes) else np.zeros((0,2))
    distances=np.linalg.norm(centroids-current_centroid,axis=1) if len(centroids) else np.asarray([])
    nearby=sorted(float(z) for z in distances if z>1)
    bbox_ious=[]
    for bx,by,bw,bh in other_boxes:
        if abs(bx-x)+abs(by-y)+abs(bw-w)+abs(bh-h)<.001:continue
        iw=max(0,min(x+w,bx+bw)-max(x,bx));ih=max(0,min(y+h,by+bh)-max(y,by))
        bbox_ious.append(iw*ih/max(1,w*h+bw*bh-iw*ih))
    raw_g=raw[SRC_HI];raw_t=raw[SRC_TILES]
    gi,gc,gmax=matching_scores(rr,raw_g);ti,tc,tmax=matching_scores(rr,raw_t)
    flat=dict(log_perimeter=np.log1p(perimeter),log_convex_area=np.log1p(hull_area),
        solidity=area/max(1,hull_area),extent=area/max(1,w*h),elongation=min(100,elong),
        eccentricity=ecc,aspect_absolute=max((w+1)/(h+1),(h+1)/(w+1)),area_frac_box=area/max(1,w*h),
        edge_density=perimeter/max(1,area),endpoint_count=endpoints,branchpoint_count=branches,
        skeleton_length=int(sk.sum())/max(.15,ratio),skeleton_area_ratio=float(sk.sum())/max(1,reduced.sum()),
        mean_thickness=float(np.mean(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        max_thickness=float(np.max(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        width_variation=float(np.std(dist)) if len(dist) else 0.,local_component_count=max(0,components),
        intensity_inside_std=float(inside.std())/255.,intensity_inside_p10=float(np.percentile(inside,10))/255.,
        intensity_inside_p50=float(np.median(inside))/255.,intensity_inside_p90=float(np.percentile(inside,90))/255.,
        intensity_ring_p50=float(np.median(outside))/255.,
        intensity_diff_p10=float(np.percentile(outside,10)-np.percentile(inside,10))/255.,
        intensity_diff_mean=float(outside.mean()-inside.mean())/255.,inside_entropy=image_entropy(inside),
        sobel_inside_mean=float(sob[mask>0].mean())/255. if area else 0.,
        sobel_ring_mean=float(sob[ring>0].mean())/255. if ring.any() else 0.,
        contrast_normalized=float(outside.mean()-inside.mean())/max(10.,float(outside.std())),
        sun_radial=rad,rim_distance=max(0,1-rad),mask_near_solar_limb=float(rad>.93),
        hull_vs_mask=hull_area/max(1,area),compactness=perimeter**2/max(1,4*np.pi*area),
        hu_1=float(np.sign(hu[0])*np.log1p(abs(hu[0]))),hu_2=float(np.sign(hu[1])*np.log1p(abs(hu[1]))),
        pca_angle_sin=float(major[1]),pca_angle_cos=float(major[0]),
        neighbor_center_dist=nearby[0]/2048 if nearby else 1.,
        neighbor_bbox_iou=max(bbox_ious,default=0.),
        neighbor_area_ratio=float(sorted([abs(np.log((w*h+1)/(bw*bh+1))) for bx,by,bw,bh in other_boxes])[1]) if len(other_boxes)>1 else 0.,
        n_neighbors_100=sum(v<100 for v in nearby),curve_origin=float(source_kind=='curve'),
        curve_conf=float(curve_meta.get('score',0)),curve_center_support=float(curve_meta.get('center_support',0)),
        curve_vs_global_iou=gi,curve_vs_tile_iou=ti,
        raw_global_count=len(raw_g),raw_tile_count=len(raw_t),
        raw_high_conf_global_count=sum(p['score']>.4 for p in raw_g),
        raw_high_conf_tile_count=sum(p['score']>.4 for p in raw_t),
        mask_global_rank=gc,mask_tile_rank=tc,mask_area_over_disk=area/(np.pi*1024**2),
        intensity_outside_std=float(outside.std())/255.)
    assert set(flat)==set(EXTRA_FIELDS),('Missing feature',set(EXTRA_FIELDS)-set(flat))
    return flat

def extended_features(stem,masks,origins=None,curve_meta=None,gray=None,raw=None):
    if not masks:return pd.DataFrame(columns=NEW_FEATURES)
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
    if raw is None:raw=load_base(stem)[0]
    # Refinement-based V24 features need the frozen, correct V24 refiner.
    if origins is None:origins=['detector']*len(masks)
    if curve_meta is None:curve_meta=[{} for _ in masks]
    assert len(masks)==len(origins)==len(curve_meta)
    refined=v24_refined_masks(stem,masks,gray) if all(o=='detector' for o in origins) else [
        run_refiner(refiner_v24,gray,r) if o=='detector' else r for r,o in zip(masks,origins)]
    base=features_v24(stem,masks,refined,gray,raw)
    boxes=[list(map(float,coco.toBbox(rr))) for rr in masks]
    for i,rr in enumerate(masks):
        extra=extract_extra(gray,rr,boxes,origins[i],curve_meta[i],raw)
        for k in EXTRA_FIELDS:base.loc[i,k]=extra[k]
    return base[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)

print('Extended feature count:',len(NEW_FEATURES))

In [ ]:
VAL_STEMS=sorted(VALIDATION)
assert len(VAL_STEMS)==104 or not FULL_VALIDATION_REQUIRED
by_day=defaultdict(list)
for st in VAL_STEMS:by_day[st[:8]].append(st)
all_days=sorted(by_day);random.Random(SEED+952).shuffle(all_days)
CAL_DAYS=set(all_days[:int(len(all_days)*.52)])
CAL=sorted(s for s in VAL_STEMS if s[:8] in CAL_DAYS)
AUDIT=sorted(s for s in VAL_STEMS if s[:8] not in CAL_DAYS)
assert len(CAL)>30 and len(AUDIT)>30 and not ({s[:8] for s in CAL}&{s[:8] for s in AUDIT})
print('Train quality on',len(CAL),'scenes, evaluate on',len(AUDIT),'different-day scenes')

# Cache masks and features separately from the checkpointed models.
def scene_candidates(stem,include_curve=True):
    raw,base=load_base(stem)
    if include_curve:
        full,origins=extend_with_curve(stem,base,raw)
    else:full,origins=base,['detector']*len(base)
    curve_list=curve_proposals(stem) if include_curve else []
    meta=[]
    for index,rr in enumerate(full):
        if origins[index]=='detector':meta.append({})
        else:
            index_ious=np.asarray(coco.iou([rr],[as_rle(v['counts']) for v in curve_list],
                                             [0]*len(curve_list)),float).reshape(-1) if curve_list else np.asarray([])
            meta.append(curve_list[int(index_ious.argmax())] if len(index_ious) else {})
    return raw,full,origins,meta,len(base)

def scene_feature_data(stem):
    path=OUT/'reports'/('features_'+stem+'.joblib')
    # Data are caches of MODEL PREDICTIONS, not ground-truth test labels.
    signature=hashlib.sha256((CURVE_SHA+REAL_REFINER_SHA+sha(V24_REFINER_PATH)+str(NEW_FEATURES)).encode()).hexdigest()
    if path.exists():
        try:
            obj=joblib.load(path)
            if obj.get('signature')==signature:return obj
        except Exception:pass
    raw,masks,origins,meta,n_base=scene_candidates(stem)
    feats=extended_features(stem,masks,origins,meta,raw=raw)
    obj={'signature':signature,'masks':[txt(x) for x in masks], 'origins':origins,
         'n_base':n_base,'features':feats,'stem':stem}
    joblib.dump(obj,path,compress=3)
    return obj

def consensus_quality_labels(stem,preds):
    expert_versions=gt(stem);out=[]
    for rr in preds:
        votes=[]
        for truth in expert_versions:
            if truth:
                vec=np.asarray(coco.iou([rr],truth,[0]*len(truth)),float).reshape(-1)
                votes.append(float(np.max(vec)) if len(vec) else 0.)
            else:votes.append(0.)
        out.append(int(np.mean(np.asarray(votes)>.5)>=.5) if votes else 0)
    return out


assert list(v25_ranker_package['columns'])==NEW_FEATURES,'V25 feature order mismatch'


## 8. Train the V26 refiner on a larger bank of **real YOLO errors**

Compared with V25's 693 examples (98.3% positive), this bank includes more low-IoU and false-positive proposals from the raw inference pool. Negatives use empty target masks, so the refiner must learn to suppress bad prompts. Positives are matched to training-only expert annotations. A **scene-separated holdout** selects the best checkpoint. This model is trained solely on the 603 historical detector-training scenes; it does not see the 104 tuning scene labels or any test labels.

In [ ]:
from torch.utils.data import DataLoader,Dataset
BANK=OUT/'real_pairs'
for part in ('image','prompt','target'):(BANK/part).mkdir(exist_ok=True)
INDEX=BANK/'pairs.csv'
if INDEX.is_file():
    bank_rows=pd.read_csv(INDEX).to_dict('records')
else:bank_rows=[]
known={str(r['key']) for r in bank_rows}
train_stems=sorted(set(TRAIN)-set(VALIDATION))
random.Random(SEED+881).shuffle(train_stems)
assert len(train_stems)==603

def write_pair(stem,key,rr,gt_rr,gray,quality,origin):
    roi=normalized_bbox_roi(coco.toBbox(rr),scale=1.60)
    if roi is None:return None
    x0,y0,x1,y1=roi
    pred=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    target=(coco.decode(gt_rr)[y0:y1,x0:x1].astype(np.uint8) if gt_rr is not None
            else np.zeros_like(pred))
    image=cv2.resize(gray[y0:y1,x0:x1],(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_AREA)
    prompt=cv2.resize(pred,(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_NEAREST)
    mask=cv2.resize(target,(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_NEAREST)
    if int(prompt.sum())<15:return None
    cv2.imwrite(str(BANK/'image'/(key+'.jpg')),image,[cv2.IMWRITE_JPEG_QUALITY,93])
    cv2.imwrite(str(BANK/'prompt'/(key+'.png')),prompt*255)
    cv2.imwrite(str(BANK/'target'/(key+'.png')),mask*255)
    return {'key':key,'stem':stem,'iou_to_expert':quality,'positive':int(gt_rr is not None),'origin':origin}

for stem in tqdm(train_stems[:V26_TRAIN_SCENES],desc='Create V26 real detector prompts'):
    if len(bank_rows)>=V26_MAX_TRAIN_PAIRS:break
    raw,accepted=load_base(stem)
    expert=[rr for version in gt(stem) for rr in version]
    if not expert:continue
    # Diverse sources: accepted masks plus moderately scored raw candidates from each detector.
    proposals=[('selected',p) for p in accepted]
    for source in (SRC_HI,SRC_TILES):
        ranked=sorted(raw[source],key=lambda x:-float(x['score']))
        proposals.extend([(source[1],as_rle(p['counts'])) for p in ranked[:28]
                          if int(p['area'])>=180 and float(p['score'])>=.12])
    if not proposals:continue
    gray=cv2.imread(str(TRAIN[stem]),cv2.IMREAD_GRAYSCALE)
    proposal_rles=[p for _,p in proposals]
    ious=np.asarray(coco.iou(proposal_rles,expert,[0]*len(expert)),float)
    # Hard cases first, but retain strong positives so pretraining isn't forgotten.
    ordering=list(range(len(proposals)))
    rng=random.Random(SEED+int(hashlib.sha256(stem.encode()).hexdigest()[:8],16))
    rng.shuffle(ordering)
    ordering.sort(key=lambda i:(0 if .12<=max(ious[i])<=.68 else (1 if max(ious[i])<.12 else 2)))
    num_neg=0;num_pos=0
    for j in ordering:
        if len(bank_rows)>=V26_MAX_TRAIN_PAIRS:break
        origin,rr=proposals[j]
        bestidx=int(ious[j].argmax());best=float(ious[j,bestidx])
        positive=best>=.12
        # Intentional balance: unlike V25, don't nearly eliminate all false-positive prompts.
        if not positive and num_neg>=5:continue
        if positive and num_pos>=10:continue
        # Very uncertain low-IoU proposals are negatives, not matched to a distant filament.
        key=f'{stem}_{origin}_{j}'
        if key in known:continue
        record=write_pair(stem,key,rr,expert[bestidx] if positive else None,gray,best,origin)
        if record is None:continue
        bank_rows.append(record);known.add(key)
        if positive:num_pos+=1
        else:num_neg+=1
    if len(bank_rows)%75<25:pd.DataFrame(bank_rows).to_csv(INDEX,index=False)
pd.DataFrame(bank_rows).to_csv(INDEX,index=False)
assert len(bank_rows)>=250,'Too few real proposal training examples'
neg_count=sum(not r['positive'] for r in bank_rows)
print('V26 training bank:',len(bank_rows),'real proposals; negatives',neg_count,'fraction',round(neg_count/len(bank_rows),3))


## 9. Fine-tune ResNet18–U-Net with **boundary-aware loss + hard-negative supervision**

Starts from the trained V25 checkpoint, not ImageNet/random weights. Uses BCE+soft Dice+edge-weighted BCE, real proposal masks, per-epoch held-scene validation, and restartable saved checkpoints. If you attach an already trained `SOLAR_V26_REFINER_BEST.pt`, the notebook skips completed epochs.

In [ ]:
class RealPairs(Dataset):
    def __init__(self,rows,train=False):self.rows=rows;self.train=train
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        record=self.rows[i];key=record['key']
        gray=cv2.imread(str(BANK/'image'/(key+'.jpg')),0)
        prompt=cv2.imread(str(BANK/'prompt'/(key+'.png')),0)
        mask=cv2.imread(str(BANK/'target'/(key+'.png')),0)
        if any(x is None for x in (gray,prompt,mask)):raise FileNotFoundError(key)
        prompt=(prompt>127).astype(np.uint8);mask=(mask>127).astype(np.uint8)
        if self.train:
            if np.random.rand()<.5:
                gray=np.fliplr(gray).copy();prompt=np.fliplr(prompt).copy();mask=np.fliplr(mask).copy()
            if np.random.rand()<.5:
                gray=np.flipud(gray).copy();prompt=np.flipud(prompt).copy();mask=np.flipud(mask).copy()
            if np.random.rand()<.25:
                gray=np.clip(gray.astype(np.float32)*np.random.uniform(.84,1.16)+np.random.uniform(-9,9),0,255).astype(np.uint8)
        return torch.from_numpy(make_channels(gray,prompt)),torch.from_numpy(mask[None].astype(np.float32))

def v26_loss(logit,target):
    prob=logit.sigmoid()
    bce=F.binary_cross_entropy_with_logits(logit,target)
    dice=1-(2*(prob*target).sum((2,3))+1)/((prob+target).sum((2,3))+1)
    # Conv filters on GPU emphasize pixels next to ground-truth boundaries.
    rim=F.max_pool2d(target,5,1,2)-(-F.max_pool2d(-target,5,1,2))
    boundary=F.binary_cross_entropy_with_logits(logit,target,reduction='none')
    edge=((1+3*rim)*boundary).mean()
    return .38*bce+.40*dice.mean()+.22*edge

unique_scene_ids=sorted({r['stem'] for r in bank_rows})
random.Random(SEED+125).shuffle(unique_scene_ids)
held=set(unique_scene_ids[:max(24,int(.17*len(unique_scene_ids)))])
train_rows=[r for r in bank_rows if r['stem'] not in held]
val_rows=[r for r in bank_rows if r['stem'] in held]
assert train_rows and val_rows and not ({r['stem'] for r in train_rows}&{r['stem'] for r in val_rows})
tr_loader=DataLoader(RealPairs(train_rows,True),batch_size=V26_BATCH,shuffle=True,num_workers=2,pin_memory=True)
va_loader=DataLoader(RealPairs(val_rows),batch_size=V26_BATCH,shuffle=False,num_workers=2,pin_memory=True)
V26_REFINER=Path('/kaggle/working/SOLAR_V26_REFINER_BEST.pt')
V26_LAST=Path('/kaggle/working/SOLAR_V26_REFINER_LAST.pt')
if not V26_REFINER.is_file():
    payload,origin=find_original_asset(V26_REFINER.name)
    if payload is not None:
        V26_REFINER.write_bytes(payload);print('Resumed V26 refiner from',origin)
refiner_v26=PromptUNet().cuda(GPU_ID)
if V26_REFINER.is_file():
    checkpoint=torch.load(V26_REFINER,map_location='cpu',weights_only=False)
    refiner_v26.load_state_dict(checkpoint['model'],strict=True)
    begin=int(checkpoint['epoch'])+1;best_val=float(checkpoint['val_loss'])
else:
    refiner_v26.load_state_dict(torch.load(V25_REAL_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
    begin=0;best_val=float('inf')
optimizer_v26=torch.optim.AdamW(refiner_v26.parameters(),lr=V26_LEARNING_RATE,weight_decay=1.5e-4)
scaler_v26=torch.amp.GradScaler('cuda')
for epoch in range(begin,V26_EPOCHS):
    refiner_v26.train();losses=[]
    for step,(x,y) in enumerate(tr_loader):
        if step>=V26_STEPS_PER_EPOCH:break
        x=x.cuda(GPU_ID,non_blocking=True);y=y.cuda(GPU_ID,non_blocking=True)
        optimizer_v26.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16):loss=v26_loss(refiner_v26(x),y)
        scaler_v26.scale(loss).backward();scaler_v26.unscale_(optimizer_v26)
        torch.nn.utils.clip_grad_norm_(refiner_v26.parameters(),1.0)
        scaler_v26.step(optimizer_v26);scaler_v26.update()
        losses.append(float(loss.detach()))
    refiner_v26.eval();vloss=[]
    with torch.inference_mode():
        for step,(x,y) in enumerate(va_loader):
            if step>=55:break
            x=x.cuda(GPU_ID,non_blocking=True);y=y.cuda(GPU_ID,non_blocking=True)
            with torch.autocast('cuda',dtype=torch.float16):vloss.append(float(v26_loss(refiner_v26(x),y)))
    score=float(np.mean(vloss))
    checkpoint={'model':{k:v.detach().cpu() for k,v in refiner_v26.state_dict().items()},
                'epoch':epoch,'val_loss':score,'real_negative_count':neg_count,
                'train_physical_scenes':len(set(r['stem'] for r in train_rows)),
                'never_trained_on_test_labels':True}
    torch.save(checkpoint,V26_LAST)
    if score<best_val:
        best_val=score;torch.save(checkpoint,V26_REFINER)
    print(f'V26 refiner: epoch {epoch} train={np.mean(losses):.5f} val={score:.5f} best={best_val:.5f}',flush=True)
assert V26_REFINER.is_file()
refiner_v26.load_state_dict(torch.load(V26_REFINER,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_v26.eval();V26_SHA=sha(V26_REFINER)
print('Frozen V26 best refiner',V26_SHA[:20]);save_json(OUT/'reports'/'refiner_training.json',
  {'new_refiner_sha':V26_SHA,'trained_on_test_labels':False,'real_pair_count':len(bank_rows),
   'negative_count':neg_count,'best_loss':best_val,'training_scenes':len(set(r['stem'] for r in train_rows))})


## 10. Assemble real V25 masks and counterfactual V26 candidates

The actual V25 recipe first ranks *detector* instances with its trained 69-feature ensemble and keeps confidence ≥0.38. It then refines those with quality ≥0.50. V26 first reproduces that control, then produces an alternative V26 mask per selected instance. No model sees test labels.

In [ ]:
def nonoverlap_after_refinement(mask_sequence):
    occupied=np.zeros((H,W),np.uint8);out=[]
    for rr in mask_sequence:
        pix=coco.decode(rr).astype(np.uint8)&(occupied==0)
        if int(pix.sum())<80:continue
        out.append(enc(pix));occupied|=pix
    return out

@torch.inference_mode()
def counterfactual_masks(stem):
    info=scene_feature_data(stem)
    p=rank_probs(new_rankers,info['features'])
    all_masks=[as_rle(c) for c in info['masks']]
    selected=[i for i in range(info['n_base']) if p[i]>=V25_QUALITY_CUTOFF]
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
    # Cache counterfactuals with model and feature hashes, not merely stem.
    signature=hashlib.sha256((V26_SHA+REAL_REFINER_SHA+sha(V25_QUALITY_PATH)+CURVE_SHA+str(V25_QUALITY_CUTOFF)).encode()).hexdigest()
    cache=OUT/'action_cache'/(stem+'.joblib')
    if cache.exists():
        try:
            rec=joblib.load(cache)
            if rec.get('sig')==signature:return rec
        except (OSError,EOFError,ValueError,KeyError):pass
    original=[all_masks[i] for i in selected]
    selected_prob=np.asarray([p[i] for i in selected],float)
    older=[real_refine(stem,rr,gray) if selected_prob[j]>=.50 else rr
           for j,rr in enumerate(original)]
    newer=[run_refiner(refiner_v26,gray,rr,threshold=.5,min_prompt_iou=.48)
           for rr in original]
    features=[]
    for k,(rr,old,new) in enumerate(zip(original,older,newer)):
        area=float(coco.area(rr));ao=float(coco.area(old));an=float(coco.area(new))
        old_new=float(coco.iou([old],[new],[0])[0,0]);original_new=float(coco.iou([rr],[new],[0])[0,0])
        original_old=float(coco.iou([rr],[old],[0])[0,0])
        x,y,w,h=map(float,coco.toBbox(rr))
        features.append({'v25_quality':float(selected_prob[k]),'logarea':float(np.log1p(area)),
          'aspect':float(np.log((w+1)/(h+1))),
          'old_new_iou':old_new,'raw_new_iou':original_new,'raw_old_iou':original_old,
          'new_area_ratio':an/max(1,ao),'old_area_ratio':ao/max(1,area),
          'new_area_change':abs(an-ao)/max(1,ao),
          'bbox_center_x':(x+w/2)/W,'bbox_center_y':(y+h/2)/H,
          'local_darkness':float(info['features'].iloc[selected[k]]['contrast']),
          'model_agreement':float(info['features'].iloc[selected[k]]['global_agreement']),
          'tile_agreement':float(info['features'].iloc[selected[k]]['tile_agreement'])})
    rec={'sig':signature,'selected':selected,'q':selected_prob,
         'original':[txt(x) for x in original],'v25':[txt(x) for x in older],
         'v26':[txt(x) for x in newer],'action_features':pd.DataFrame(features)}
    joblib.dump(rec,cache,compress=3)
    return rec

ACTIONS=['v25_quality','logarea','aspect','old_new_iou','raw_new_iou','raw_old_iou','new_area_ratio',
 'old_area_ratio','new_area_change','bbox_center_x','bbox_center_y','local_darkness','model_agreement','tile_agreement']

def rec_rles(rec,name):return [as_rle(s) for s in rec[name]]

def v25_selected(stem):
    rec=counterfactual_masks(stem)
    return nonoverlap_after_refinement(rec_rles(rec,'v25'))
print('Counterfactual generator initialized; frozen V25 cutoff',V25_QUALITY_CUTOFF)


## 11. Train an **action selector** on calibration days, judge it on different dates

For every V25-accepted instance, compare the existing V25-refined mask with the V26 candidate against training-only expert annotations. A group-cross-fitted classifier learns when V26 improves matched IoU without crossing the PQ match threshold in the wrong direction. Selective replacement thresholds are chosen from calibration OOF predictions only. The 51-date audit is not used for model training/threshold tuning.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier,RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

def oracle_quality_for_instance(stem,rr):
    # Average best IoU per independent annotator; no test labels in this function.
    values=[]
    for expert in gt(stem):
        if expert:
            overlaps=np.asarray(coco.iou([rr],expert,[0]*len(expert)),float).reshape(-1)
            values.append(float(np.max(overlaps)) if overlaps.size else 0.)
        else:values.append(0.)
    return float(np.mean(values)) if values else 0.

cal_data=[];labels=[];groups=[];gains=[]
for stem in tqdm(CAL,desc='Build genuine counterfactual oracle labels'):
    rec=counterfactual_masks(stem)
    before=rec_rles(rec,'v25');after=rec_rles(rec,'v26')
    f=rec['action_features']
    assert len(before)==len(after)==len(f)
    for idx,(old,new) in enumerate(zip(before,after)):
        before_iou=oracle_quality_for_instance(stem,old)
        after_iou=oracle_quality_for_instance(stem,new)
        gain=after_iou-before_iou
        # Do not reward an improved pixel score if it destroys the >=.5 TP correspondence.
        label=int(gain>=.012 and not (before_iou>=.5 and after_iou<=.5))
        cal_data.append(f.iloc[idx].to_dict());labels.append(label);groups.append(stem[:8]);gains.append(gain)
AX=pd.DataFrame(cal_data,columns=ACTIONS).replace([np.inf,-np.inf],0).fillna(0.)
ay=np.asarray(labels,int);ag=np.asarray(groups)
print('Counterfactual examples:',len(ay),'beneficial corrections:',int(ay.sum()),'positive fraction',round(ay.mean(),3) if len(ay) else None)
ACTION_ACTIVE=(len(ay)>=120 and ay.sum()>=15 and (len(ay)-ay.sum())>=30 and len(set(ag))>=4)
ACTION_PATH=Path('/kaggle/working/SOLAR_V26_ACTION_SELECTOR.joblib')
THRESHOLD_GRID=[.40,.50,.60,.70,.80]

def action_model():
    return HistGradientBoostingClassifier(max_iter=95,learning_rate=.045,max_leaf_nodes=7,
                 min_samples_leaf=14,l2_regularization=6.0,random_state=SEED+26)

if ACTION_ACTIVE:
    folds=GroupKFold(n_splits=4);oof=np.full(len(ay),np.nan)
    for ti,vi in folds.split(AX,ay,ag):
        if len(set(ay[ti]))<2:
            ACTION_ACTIVE=False;break
        learner=action_model();learner.fit(AX.iloc[ti],ay[ti])
        oof[vi]=learner.predict_proba(AX.iloc[vi])[:,1]
if ACTION_ACTIVE:
    # Decision threshold optimized against actual pooled PQ on CAL OOF, not label accuracy.
    # Map per-scene predictions to continuous OOF row ranges.
    by_cal={};offset=0
    for stem in CAL:
        n=len(counterfactual_masks(stem)['selected']);by_cal[stem]=(offset,offset+n);offset+=n
    def choose_oof_action(t):
        total=np.zeros(4,float)
        for stem,(a,b) in by_cal.items():
            rec=counterfactual_masks(stem)
            old=rec_rles(rec,'v25');new=rec_rles(rec,'v26')
            pred=[n if prob>=t else o for o,n,prob in zip(old,new,oof[a:b])]
            total+=physical_counts(stem,nonoverlap_after_refinement(pred))
        return pq(total)
    action_oof_scores={float(t):choose_oof_action(t) for t in THRESHOLD_GRID}
    baseline_oof_score=sum(0 for _ in [])  # baseline computed next for exact same CAL subset
    baseline_counts=np.zeros(4,float)
    for stem in CAL:baseline_counts+=physical_counts(stem,v25_selected(stem))
    baseline_oof_score=pq(baseline_counts)
    chosen=max(action_oof_scores,key=lambda t:(action_oof_scores[t],t))
    # A selector must actually outperform V25 on calibration OOF; otherwise disable its modifications.
    if action_oof_scores[chosen]<baseline_oof_score+.0005:
        ACTION_ACTIVE=False
    else:
        gate_model=action_model().fit(AX,ay)
        joblib.dump({'model':gate_model,'columns':ACTIONS,'cutoff':chosen,
                     'calibration_days':sorted(CAL_DAYS),'calibration_oof_auc':float(roc_auc_score(ay,oof)),
                     'train_test_label_leakage':False},ACTION_PATH)
        print('Counterfactual action cutoff:',chosen,'CAL OOF PQ:',action_oof_scores,'baseline',baseline_oof_score)
if not ACTION_ACTIVE:
    chosen=1.01 # No learned actions, V25 reconstruction is unchanged.
    gate_model=None
    action_oof_scores={};print('Counterfactual action learner disabled: insufficient or nonimproving OOF evidence.')
ACTION_CUTOFF=chosen
save_json(OUT/'reports'/'action_model_report.json',{
   'active':ACTION_ACTIVE,'examples':len(ay),'positive_changes':int(ay.sum()),
   'oof_pq':action_oof_scores,'action_cutoff':ACTION_CUTOFF,
   'trained_on_calibration_only':True,'test_labels_used':False})


## 12. Per-module audit: the exact V25 winner, V26 refiner alone, selective action, conservative action

In [ ]:
# No curve proposal is added to the winner. It is needed ONLY to reproduce the V25 ranker's context.
ABLATIONS={
 'V25_PROVEN_039':{'mode':'v25'},
 'V26_REFINER_ALL':{'mode':'v26_all'},
 'V26_REFINER_HIGHQ':{'mode':'highq'},
 'V26_SELECTIVE_ACTION':{'mode':'selective'},
 'V26_SELECTIVE_STRICT':{'mode':'strict'},
 'V26_V25_V26_INTERSECTION':{'mode':'intersection'},
 'V26_V25_V26_UNION':{'mode':'union'},
}

def make_variant(stem,name,rec=None):
    rec=rec if rec is not None else counterfactual_masks(stem)
    original=rec_rles(rec,'original');v25=rec_rles(rec,'v25');v26=rec_rles(rec,'v26')
    q=rec['q'];mode=ABLATIONS[name]['mode']
    action_p=np.zeros(len(v25),float)
    if ACTION_ACTIVE and len(v25):
        action_p=gate_model.predict_proba(rec['action_features'][ACTIONS].fillna(0.))[:,1]
    masks=[]
    for i,(rr,old,new) in enumerate(zip(original,v25,v26)):
        if mode=='v25':candidate=old
        elif mode=='v26_all':candidate=new
        elif mode=='highq':candidate=new if q[i]>=.65 else old
        elif mode=='selective':candidate=new if ACTION_ACTIVE and action_p[i]>=ACTION_CUTOFF else old
        elif mode=='strict':candidate=new if ACTION_ACTIVE and action_p[i]>=max(.75,ACTION_CUTOFF) else old
        elif mode=='intersection':
            candidate=coco.merge([old,new],intersect=True) if float(coco.iou([old],[new],[0])[0,0])>=.60 else old
        elif mode=='union':
            iou=float(coco.iou([old],[new],[0])[0,0])
            merged=coco.merge([old,new]) if iou>=.78 else old
            candidate=merged if float(coco.area(merged))<=1.25*float(coco.area(old)) else old
        else:raise ValueError(mode)
        masks.append(candidate)
    return nonoverlap_after_refinement(masks)

AUDIT_SCORES={k:np.zeros(4,float) for k in ABLATIONS}
AUDIT_DETAILS=[]
for stem in tqdm(AUDIT,desc='V26 acquisition-day audit PQ'):
    rec=counterfactual_masks(stem)
    for name in ABLATIONS:
        predicted=make_variant(stem,name,rec)
        count=physical_counts(stem,predicted)
        AUDIT_SCORES[name]+=count
        AUDIT_DETAILS.append({'stem':stem,'day':stem[:8],'method':name,'sum_iou':count[0],
                              'TP':count[1],'FP':count[2],'FN':count[3], 'instances':len(predicted)})
control_name='V25_PROVEN_039'
audit_baseline=pq(AUDIT_SCORES[control_name])
board=pd.DataFrame([{'method':k,**metrics(v),'delta_audit_vs_V25':pq(v)-audit_baseline}
                    for k,v in AUDIT_SCORES.items()]).sort_values('PQ',ascending=False)
board.to_csv(OUT/'reports'/'V26_AUDIT_PQ.csv',index=False)
pd.DataFrame(AUDIT_DETAILS).to_csv(OUT/'reports'/'V26_AUDIT_PER_SCENE.csv',index=False)
display(board)
assert abs(audit_baseline-.444146)<.028,('V25 reconstructed baseline unexpected',audit_baseline)

# Grouped bootstrap (uncertainty under repeated acquisition-day sampling, not private score).
day_matrix=pd.DataFrame(AUDIT_DETAILS).groupby(['method','day'])[['sum_iou','TP','FP','FN']].sum()
all_audit_days=sorted({x['day'] for x in AUDIT_DETAILS})
mat={k:np.stack([day_matrix.loc[(k,d)].to_numpy(float) for d in all_audit_days]) for k in ABLATIONS}
ix=np.random.default_rng(SEED+26).integers(0,len(all_audit_days),size=(1100,len(all_audit_days)))
def batch_pq(arr):
    den=arr[:,1]+.5*(arr[:,2]+arr[:,3])
    return np.divide(arr[:,0],den,out=np.zeros(len(arr)),where=den>0)
baseboot=batch_pq(mat[control_name][ix].sum(axis=1));uncertainty=[]
for k in ABLATIONS:
    if k==control_name:continue
    delta=batch_pq(mat[k][ix].sum(axis=1))-baseboot
    uncertainty.append({'method':k,'delta_audit':pq(AUDIT_SCORES[k])-audit_baseline,
         'bootstrap_2p5':float(np.quantile(delta,.025)),'bootstrap_97p5':float(np.quantile(delta,.975)),
         'bootstrap_positive_fraction':float(np.mean(delta>0))})
boot=pd.DataFrame(uncertainty).sort_values('delta_audit',ascending=False)
boot.to_csv(OUT/'reports'/'V26_GROUPED_BOOTSTRAP.csv',index=False)
display(boot)
# Conservative automatic selection; no need to burn public attempts on unconvincing local changes.
eligible=boot[(boot.delta_audit>=.004)&(boot.bootstrap_positive_fraction>=.90)]
SELECTED=str(eligible.iloc[0].method) if len(eligible) else control_name
print('V26 selected Kaggle default:',SELECTED,'| V25 known public 0.39, new public unknown')
save_json(OUT/'reports'/'V26_selection.json',{'chosen':SELECTED,'confirmed_public_V25':.39,
  'full_test_public_V26':None,'historical_validation_reused':True,
  'audit_score':{k:pq(v) for k,v in AUDIT_SCORES.items()},'test_ground_truth_used':False})


## 13. Generate all 180 real test predictions; preserve **byte-identical V25 0.39**

If the exact historic V25 `submission.csv` is in your archive, we recover and protect those bytes (known SHA). We also regenerate the V25 winner from the original checkpointed models and compare hashes. New candidates are saved separately even if the audit gate falls back to V25.

In [ ]:
import csv
TEST_STEMS=sorted(TEST)
assert len(TEST_STEMS)==180
# Protect the exact historical file from the previous successfully scored Kaggle run.
PROVEN=Path('/kaggle/working/V26_PROVEN_V25_039.csv')
found_previous=False
for requested in ('submission.csv','V25_REAL_REFINE_PLUS_QUALITY.csv'):
    data,origin=find_original_asset(requested)
    if data is not None and hashlib.sha256(data).hexdigest().startswith(V25_EXPECTED_CSV_SHA):
        PROVEN.write_bytes(data);found_previous=True
        print('Loaded exact 0.39 V25 Kaggle CSV bytes from',origin)
        break
if V26_REQUIRE_OLD_WINNER and not found_previous:
    print('Exact V25 CSV not separately attached; regenerate and verify the reference SHA below.')

for stem in tqdm(TEST_STEMS,desc='V26 real test image models + feature caches'):
    counterfactual_masks(stem)

PATHS={k:OUT/'submissions'/(k+'.csv') for k in ABLATIONS}
count_records=[]
from contextlib import ExitStack
with ExitStack() as stack:
    w={k:csv.writer(stack.enter_context(open(path,'w',newline='',encoding='utf8'))) for k,path in PATHS.items()}
    for writer in w.values():writer.writerow(['filament_id','segmentation_rle'])
    for stem in tqdm(TEST_STEMS,desc='Write all complete V26 Kaggle predictions'):
        rec=counterfactual_masks(stem)
        for name,writer in w.items():
            masks=make_variant(stem,name,rec)
            count_records.append({'stem':stem,'method':name,'instances':len(masks)})
            for i,rr in enumerate(masks,1):writer.writerow([f'{stem}_{i}',txt(rr)])

counts=pd.DataFrame(count_records);counts.to_csv(OUT/'reports'/'V26_TEST_INSTANCE_COUNTS.csv',index=False)
manifest={}
for name,path in PATHS.items():
    data=pd.read_csv(path,dtype=str)
    assert list(data.columns)==['filament_id','segmentation_rle'] and len(data)>200
    assert data.filament_id.is_unique and data.segmentation_rle.notna().all()
    assert data.filament_id.str.rsplit('_',n=1).str[0].isin(TEST).all()
    assert counts[counts.method==name].stem.nunique()==180
    assert int(counts[counts.method==name].instances.sum())==len(data)
    for value in data.segmentation_rle:assert int(coco.area(as_rle(value)))>0
    manifest[name]={'rows':len(data),'sha256':sha(path),'path':str(path)}
    print('TEST',name,manifest[name]['rows'],'rows',manifest[name]['sha256'][:18])

# If the regenerated reference is not byte-identical, don't call it a known-public-score CSV.
regen_sha=sha(PATHS[control_name]);regen_matches=regen_sha.startswith(V25_EXPECTED_CSV_SHA)
if regen_matches:
    shutil.copy2(PATHS[control_name],PROVEN)
    found_previous=True
assert not V26_REQUIRE_OLD_WINNER or found_previous, (
 'V25 winner could not be recovered byte-identically. Attach SOLAR_V25_ALL_PREDICTIONS_AND_REPORTS.zip '
 'or the V25 successful submission.csv alongside your model archive.')
assert not found_previous or sha(PROVEN).startswith(V25_EXPECTED_CSV_SHA)
if found_previous:
    assert len(pd.read_csv(PROVEN))==V25_EXPECTED_ROWS

MAIN=Path('/kaggle/working/submission.csv')
if SELECTED==control_name:
    shutil.copy2(PROVEN,MAIN)
else:
    shutil.copy2(PATHS[SELECTED],MAIN)
for rank,name in enumerate([x for x in ABLATIONS if x!=control_name],1):
    shutil.copy2(PATHS[name],Path('/kaggle/working')/f'V26_ALT_{rank}.csv')
save_json(OUT/'reports'/'V26_manifest.json',{'public_best_confirmed':.39,'target':.40,
  'new_public_score':None,'default_method':SELECTED,'exact_v25_reference_found':found_previous,
  'v25_reference_sha_matches':regen_matches,'reference_path':str(PROVEN),'candidates':manifest,
  'new_refiner_sha':V26_SHA,'action_active':ACTION_ACTIVE,'test_labels_accessed':False})
print('Default Kaggle submission:',MAIN,'| selection:',SELECTED)


## 14. Pack an actual reusable model bundle and all candidate CSV/report files

In [ ]:
REPORT_ZIP=Path('/kaggle/working/SOLAR_V26_PREDICTIONS_REPORTS.zip')
with zipfile.ZipFile(REPORT_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3,allowZip64=True) as z:
    for p in sorted((OUT/'reports').glob('*.csv'))+sorted((OUT/'reports').glob('*.json')):
        z.write(p,arcname='reports/'+p.name)
    for p in sorted((OUT/'submissions').glob('*.csv')):
        z.write(p,arcname='submissions/'+p.name)
    z.write('/kaggle/working/submission.csv',arcname='submission.csv')
    z.write(PROVEN,arcname=PROVEN.name)
with zipfile.ZipFile(REPORT_ZIP) as z:assert z.testzip() is None
MODEL_ZIP=Path('/kaggle/working/SOLAR_V26_ALL_TRAINED_CHECKPOINTS.zip')
with zipfile.ZipFile(MODEL_ZIP,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    for p in WEIGHTS.values():z.write(p,arcname='detectors/'+p.name)
    for p in (V24_REFINER_PATH,V24_QUALITY_PATH,V25_REAL_PATH,V25_CURVE_PATH,V25_QUALITY_PATH,V26_REFINER,V26_LAST):
        if p.is_file():z.write(p,arcname='models/'+p.name)
    if ACTION_ACTIVE and ACTION_PATH.is_file():z.write(ACTION_PATH,arcname='models/'+ACTION_PATH.name)
    z.write(REPORT_ZIP,arcname=REPORT_ZIP.name)
with zipfile.ZipFile(MODEL_ZIP) as z:assert z.testzip() is None
print('FINISHED',MAIN)
print('CSV + reports:',REPORT_ZIP,'MB',round(REPORT_ZIP.stat().st_size/2**20,2))
print('Complete trained checkpoints:',MODEL_ZIP,'MB',round(MODEL_ZIP.stat().st_size/2**20,2))
print('Known Kaggle best 0.39; a new 0.40 is NOT asserted. Review audit and selected variant before public submission.')


## Interpreting results

Look at `solar_v26/reports/V26_AUDIT_PQ.csv` and `V26_GROUPED_BOOTSTRAP.csv`: **V25_PROVEN_039** is the baseline. `V26_REFINER_ALL` tests whether hard-negative refinement helps unconditionally; `V26_SELECTIVE_ACTION` and `V26_SELECTIVE_STRICT` use the new learned action gate; the other files are controlled geometric alternatives. **Graph merging and independent curve additions are intentionally *not* reintroduced** because V25 observed inadequate graph supervision and curve proposals did not help its audit. V25's curve checkpoint remains necessary to construct its actual quality features.

The final `submission.csv` is only changed from the protected V25 file when a new method clears the audit/paired-bootstrap gates. Kaggle must score a new file before any claim of beating public **0.39**. All train labels used come from the official 707 labeled training images; the 180 test images are used for inference only.
